# Messages and Structured Output

**What you will build:** the skill of getting a model to return **typed, validated data** — a Python object you can trust — instead of free-form prose. This is the difference between a demo and something you can put in a program.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/02_messages_and_structured_output.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run this once per notebook.
!pip install -q openai pydantic

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"   # change to any model on openrouter.ai/models
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready. Model:", MODEL)

## The problem with free text

In 0.1 the model returned a paragraph. That's fine to *read*, but impossible to *program against*. If you ask "classify this support ticket," you don't want three sentences — you want a field you can branch on:

```
free text:      "Well, this looks like it's probably a billing issue, though..."   ← unparseable
structured:     { "category": "billing", "urgency": "high" }                    ← usable
```

Getting that second form reliably is **structured output**, and it's the foundation the rest of the course stands on (tool arguments, agent state, evals — all structured). Let's see the naive way first, watch it break, then fix it.

## Attempt 1 — just ask for JSON (and see why it's not enough)

The obvious move: tell the model to reply with JSON.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify the support ticket. Reply with JSON: category and urgency."},
        {"role": "user",   "content": "I was charged twice for my subscription this month!"},
    ],
    temperature=0,
)
print(resp.choices[0].message.content)

In [ ]:
# One run can get lucky. Run the same request 5 times at temperature=1
# (production traffic is rarely temperature=0) and count the failures:
failures = 0
for i in range(5):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Classify the support ticket. Reply with JSON: category and urgency."},
            {"role": "user",   "content": "I was charged twice for my subscription this month!"},
        ],
        temperature=1,
    )
    text = r.choices[0].message.content
    try:
        json.loads(text)
        print(f"run {i+1}: parsed OK")
    except json.JSONDecodeError:
        failures += 1
        print(f"run {i+1}: NOT valid JSON -> {text[:70]!r}")

print(f"\n{failures}/5 responses failed to parse")

Maybe all five parsed today — with another model, another provider behind OpenRouter, or a longer prompt, some won't. Typical failure modes: JSON wrapped in ```` ```json ```` fences, a "Sure, here's the classification:" preamble, or a field named `priority` instead of `urgency`. **You can't build on "often."** Two fixes stack on top of each other: force JSON at the API level, and validate the shape with Pydantic.

## Fix 1 — force JSON with `response_format`

Most providers support a JSON mode that guarantees the response is parseable JSON (no prose, no fences).

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify the support ticket. Reply with JSON keys: category, urgency."},
        {"role": "user",   "content": "I was charged twice for my subscription this month!"},
    ],
    response_format={"type": "json_object"},   # ← guarantees valid JSON text
    temperature=0,
)
data = json.loads(resp.choices[0].message.content)   # now this always parses
print(type(data), data)

## Fix 2 — validate the *shape* with Pydantic

JSON mode guarantees *valid JSON*, not *the JSON you wanted*. The model might still omit a field or return `urgency: "very high"` when you only allow three levels. [Pydantic](https://docs.pydantic.dev/) is the standard Python library for exactly this: declare the shape as a class, and it validates and coerces the data, raising a clear error when it doesn't fit.

To see the safety net actually *catch* something, we'll first validate the dict from Fix 1 against a schema that demands **more than we asked for** — the prompt requested two keys, the schema below requires three:

```{note}
Pydantic is worth learning on its own — it's a dependency of a huge slice of the Python ecosystem, and the whole of **PydanticAI** (Block 1) is built on it. What you do by hand here, PydanticAI will do for you automatically.
```

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    urgency:  Literal["low", "medium", "high"]
    summary:  str = Field(description="One-line summary of the issue")

# The dict from Fix 1 has category and urgency — but this schema also
# demands a summary. Valid JSON, wrong shape. Watch Pydantic catch it:
try:
    ticket = TicketClassification.model_validate(data)
    print("The model happened to volunteer all three fields:", ticket)
except ValidationError as e:
    print("ValidationError — this is the safety net working:\n")
    print(e)

**Valid JSON ≠ the JSON you wanted.** JSON mode guaranteed the response *parses*; it said nothing about which fields are inside. The schema is the contract, and Pydantic is what enforces it. The immediate fix is to ask for exactly what the schema demands:

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "Classify the support ticket. Reply with JSON keys: "
            "category (billing|technical|account|other), "
            "urgency (low|medium|high), "
            "summary (one line)."
        )},
        {"role": "user", "content": "I was charged twice for my subscription this month!"},
    ],
    response_format={"type": "json_object"},
    temperature=0,
)
data = json.loads(resp.choices[0].message.content)
ticket = TicketClassification.model_validate(data)
print(ticket)
print("category is now a real, checked field:", ticket.category)

## Putting it together: a reusable `extract` helper

Notice what just happened: we had to keep the prompt and the schema in sync *by hand* — list the fields in the system message, then repeat them in the class. That duplication is exactly what a helper should remove. The function below feeds the model the schema (generated from the class, so it can't drift), and — the important part — **when validation fails, it sends the error back to the model and lets it correct itself**. That retry loop is the pattern every agent framework implements for you; PydanticAI (Block 1) does precisely this under the hood.

In [ ]:
def extract(model_class, user_text, instructions, max_retries=2):
    """Call the LLM and return a validated instance of `model_class`.

    On ValidationError, the error text is sent back to the model and it
    gets another try — the same self-correction loop every framework
    automates.
    """
    schema = model_class.model_json_schema()
    messages = [
        {"role": "system",
         "content": f"{instructions}\nReturn JSON matching this schema:\n{json.dumps(schema)}"},
        {"role": "user", "content": user_text},
    ]
    for attempt in range(1 + max_retries):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0,
        )
        raw = resp.choices[0].message.content
        try:
            return model_class.model_validate_json(raw)
        except ValidationError as e:
            if attempt == max_retries:
                raise                      # out of retries: surface the error
            print(f"attempt {attempt + 1} failed validation, retrying...")
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user",
                             "content": f"That JSON failed validation:\n{e}\n"
                                        "Return corrected JSON matching the schema."})

result = extract(
    TicketClassification,
    "The app crashes every time I open the reports page. I need this fixed today.",
    "Classify the support ticket.",
)
print(result)

::::{dropdown} 🔧 Under the hood: what `response_format` changes
:color: info

`response_format={"type": "json_object"}` adds a constraint on the server side so the model's output is guaranteed to be syntactically valid JSON — no ```` ``` ```` fences, no "Sure, here's..." preamble. Some providers go further with `{"type": "json_schema", ...}`, which also enforces the *fields*. Support varies by model, which is exactly why we add the Pydantic validation layer on top: it makes the guarantee ours, not the provider's. That portability is the durable choice.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a field.** Give `TicketClassification` a `sentiment: Literal["angry", "neutral", "happy"]` and re-run `extract`. Notice you changed *one class* and the prompt updated itself (the schema is generated).
2. **Watch the retry heal a failure.** Call `extract` with `max_retries=0` and a schema the model tends to get wrong on the first try (e.g. add a field `word_count: int` with the instruction "word_count must equal the number of words in summary"). When it raises, restore `max_retries=2` and run the same call — the printed `attempt N failed validation, retrying...` line is the self-correction loop in action.
3. **Extract something else.** Write a `Recipe` model (`title: str`, `ingredients: list[str]`, `minutes: int`) and extract it from a paragraph of cooking text. **Done when:** `type(result.minutes) is int` — you can do arithmetic on model output.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Add a field:**

```python
class TicketClassification(BaseModel):
    category:  Literal["billing", "technical", "account", "other"]
    urgency:   Literal["low", "medium", "high"]
    summary:   str = Field(description="One-line summary of the issue")
    sentiment: Literal["angry", "neutral", "happy"]

extract(TicketClassification, "I was charged twice!", "Classify the support ticket.")
```

No prompt edit needed — `model_json_schema()` regenerates the contract from the class.

**2 — Watch the retry:** the exact failure depends on the model, but the shape of the experiment is:

```python
class StrictTicket(TicketClassification):
    word_count: int = Field(description="Must equal the number of words in summary")

# extract(StrictTicket, ..., max_retries=0)  -> often raises ValidationError
# extract(StrictTicket, ..., max_retries=2)  -> prints "attempt 1 failed..." then succeeds
```

If your model gets it right first try, make the constraint harder. The lesson is the mechanism, not the specific failure.

**3 — Recipe:**

```python
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    minutes: int

r = extract(
    Recipe,
    "Grandma's tortilla: beat 4 eggs, fry two sliced potatoes and half an "
    "onion in olive oil, combine and cook 5 minutes per side. About 30 "
    "minutes start to finish.",
    "Extract the recipe.",
)
assert type(r.minutes) is int
print(r)
```
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| Error mentioning `response_format` not supported — or JSON mode silently ignored | OpenRouter routed the request to a provider without JSON mode for this model | Switch models, or drop `response_format` entirely: the schema-in-prompt plus Pydantic validation plus the retry in `extract` already give you the guarantee — that's why we built it belt-and-braces |
| `ValidationError` even after retries | The model genuinely can't produce the shape (too many fields, exotic types, contradictory instructions) | Simplify the schema, make non-essential fields `Optional`, or try a stronger model |
| `json.JSONDecodeError` even with JSON mode on | Response truncated mid-object (`finish_reason == "length"`) | Check `resp.choices[0].finish_reason`; raise `max_tokens` |
::::

**What's next:** in **0.3** we give the model **tools** and write the agent loop by hand. Structured output is what makes it possible: a tool call is just structured output the model produces to ask for an action.